<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-03-data-extraction-from-documents/notebooks/Ophelie_SDF_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extract Text & Bounding Boxes from SDF using PyMuPDF
Downloads `sample-sdf-document.pdf`, extracts word-level text with bounding boxes from every page using `fitz.get_text("words")`, and saves a structured JSON output.

In [ ]:
!pip install -q pymupdf gdown
import fitz
import json
import gdown
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 26.6 MB/s eta 0:00:00


In [ ]:
# Download the sample SDF document provided for this task
file_id = '1yPq5BBfHhNhKDQuRN4uDyyEDF77sz_YO'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'sample_sdf_document.pdf', quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1yPq5BBfHhNhKDQuRN4uDyyEDF77sz_YO
To: /content/sample_sdf_document.pdf
100%|██████████| 147k/147k [00:00<00:00, 6.80MB/s]


'sample_sdf_document.pdf'

## Step 2: Inspect the document

In [ ]:
doc = fitz.open('sample_sdf_document.pdf')
print(f"Total Pages: {doc.page_count}")
print("PDF Metadata:")
print(doc.metadata)

Total Pages: 3
PDF Metadata:
{'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': '', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


## Step 3: Extract words with bounding boxes from every page

In [ ]:
extracted_data = []
for page_num in range(len(doc)):
    page = doc[page_num]
    words = page.get_text("words")
    for w in words:
        x0, y0, x1, y1, word, block_no, line_no, word_no = w
        extracted_data.append({
            "page": page_num + 1,
            "text": word,
            "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)],
            "block_no": block_no,
            "line_no": line_no,
            "word_no": word_no
        })

print(f"Total words extracted across {doc.page_count} pages: {len(extracted_data)}")
print("Sample entries:")
for entry in extracted_data[:10]:
    print(entry)

Total words extracted across 3 pages: 617
Sample entries:
{'page': 1, 'text': 'Cytiva', 'bbox': [547.44, 87.59, 576.01, 98.43], 'block_no': 0, 'line_no': 0, 'word_no': 0}
{'page': 1, 'text': '100', 'bbox': [508.31, 99.96, 523.84, 110.4], 'block_no': 0, 'line_no': 1, 'word_no': 0}
{'page': 1, 'text': 'Results', 'bbox': [525.84, 99.96, 556.06, 110.4], 'block_no': 0, 'line_no': 1, 'word_no': 1}
{'page': 1, 'text': 'Way', 'bbox': [558.21, 99.96, 576.02, 110.4], 'block_no': 0, 'line_no': 1, 'word_no': 2}
{'page': 1, 'text': 'Marlborough,', 'bbox': [477.12, 111.96, 532.15, 122.4], 'block_no': 0, 'line_no': 2, 'word_no': 0}
{'page': 1, 'text': 'MA', 'bbox': [534.09, 111.96, 547.87, 122.4], 'block_no': 0, 'line_no': 2, 'word_no': 1}
{'page': 1, 'text': '01752', 'bbox': [549.92, 111.96, 575.91, 122.4], 'block_no': 0, 'line_no': 2, 'word_no': 2}
{'page': 1, 'text': 'United', 'bbox': [520.43, 123.96, 547.73, 134.4], 'block_no': 1, 'line_no': 0, 'word_no': 0}
{'page': 1, 'text': 'States', 'bbox': 

## Step 4: Also capture block-level text (paragraph structure) for comparison

In [ ]:
blocks_data = []
for page_num in range(len(doc)):
    page = doc[page_num]
    blocks = page.get_text("blocks")
    for b in blocks:
        x0, y0, x1, y1, text = b[0], b[1], b[2], b[3], b[4]
        blocks_data.append({
            "page": page_num + 1,
            "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)],
            "text": text.strip()
        })

print(f"Total blocks extracted: {len(blocks_data)}")
for b in blocks_data[:3]:
    print(b)

Total blocks extracted: 58
{'page': 1, 'bbox': [477.12, 87.59, 576.02, 122.4], 'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752'}
{'page': 1, 'bbox': [520.43, 123.96, 575.95, 134.4], 'text': 'United States'}
{'page': 1, 'bbox': [295.2, 754.14, 339.68, 765.71], 'text': 'Page 1 / 1'}


## Step 5: Save structured output

In [ ]:
with open('sdf_extracted_words.json', 'w') as f:
    json.dump(extracted_data, f, indent=2)

with open('sdf_extracted_blocks.json', 'w') as f:
    json.dump(blocks_data, f, indent=2)

page_counts = Counter(d['page'] for d in extracted_data)
print("Words per page:", dict(page_counts))
print("Saved sdf_extracted_words.json and sdf_extracted_blocks.json")

Words per page: {1: 203, 2: 201, 3: 213}
Saved sdf_extracted_words.json and sdf_extracted_blocks.json
